In [ ]:
#!/usr/bin/env python3
"""
Subscribes to apoorv/time/publish and prints each message.
Uses a persistent session (clean_session=False) + QoS 1 so the broker queues
messages published while this client was offline and delivers them on reconnect.

Broker options (change BROKER/PORT below):
  Local:  localhost:1883   (brew install mosquitto && brew services start mosquitto)
  Docker: localhost:1883   (docker run -d -p 1883:1883 eclipse-mosquitto)
  Public: test.mosquitto.org  (unreliable for persistent session queuing)
"""

import paho.mqtt.client as mqtt

BROKER = "localhost"
PORT = 1883
TOPIC = "apoorv/time/publish"
QOS = 1
CLIENT_ID = "apoorv_time_subscriber"  # fixed ID required for persistent session


def on_connect(client, userdata, flags, reason_code, properties):
    if reason_code == 0:
        print(f"Connected to {BROKER} (session resumed: {flags.session_present})")
        client.subscribe(TOPIC, qos=QOS)
        print(f"Subscribed to '{TOPIC}'. Waiting for messages...\n")
    else:
        print(f"Connection failed with code {reason_code}")


def on_message(client, userdata, msg):
    print(f"[{msg.topic}] {msg.payload.decode()}")


client = mqtt.Client(
    mqtt.CallbackAPIVersion.VERSION2,
    client_id=CLIENT_ID,
    clean_session=False,
)
client.on_connect = on_connect
client.on_message = on_message

client.connect(BROKER, PORT, keepalive=60)
client.loop_forever()  # blocks; interrupt kernel to stop
